# @devops — Create GitHub Repo & Push Workspace Content

**Task:** `DEVOPS-GITHUB-CREATE` + `DEVOPS-LINK-REPO-PUSH`  
**Repo:** `skhage/provable-resilience-intelligence`  
**Status:** Local Git folder has 9 commits on `main`, DAB validates (dev ✅, prod ✅)  

## ⚠️ KNOWN ISSUE: Databricks gitHubOAuth token has EMPTY scopes
The workspace OAuth credential (`skhage-github`, ID 358347682222830) **cannot create repos** — GitHub returns 403 "Resource not accessible by integration".

## TWO OPTIONS:

### Option A — Create repo on GitHub web (fastest, 30 seconds):
1. Go to **https://github.com/new**
2. Name: `provable-resilience-intelligence`
3. Description: `Provable Resilience Intelligence — Governed resilience scoring for Lakelink Fiber enterprise services`
4. Public, **do NOT initialize with README** (we have existing commits)
5. Click "Create repository"
6. Then run **Cell 3 (Push)** below

### Option B — Use a GitHub PAT:
1. Go to https://github.com/settings/tokens/new
2. Create a classic token with `repo` scope
3. Add it as a Databricks Git credential (Settings → Linked Accounts)
4. Then run **Cell 2 (Create via API)** below

### After repo exists, run:
- Cell 3: Push 9 local commits to GitHub
- Cell 4: Create `develop` branch on GitHub
- Cell 5: Verify

In [0]:
import json
import urllib.request
import urllib.error
import subprocess

# Get GitHub OAuth token via git credential helper (workspace Git credential)
process = subprocess.Popen(
    ["git", "credential", "fill"],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
stdout, _ = process.communicate(input="protocol=https\nhost=github.com\n\n", timeout=10)
token = None
for line in stdout.strip().split("\n"):
    if line.startswith("password="):
        token = line.split("=", 1)[1]
assert token, "No token found via git credential fill"
print(f"Token obtained: {token[:4]}...{token[-4:]} (len={len(token)})")

gh_headers = {
    "Authorization": f"token {token}",
    "Accept": "application/vnd.github+json",
    "User-Agent": "databricks-devops",
    "X-GitHub-Api-Version": "2022-11-28"
}

# First verify the token works — check authenticated user
try:
    user_req = urllib.request.Request("https://api.github.com/user", headers=gh_headers)
    user_resp = urllib.request.urlopen(user_req, timeout=15)
    user_data = json.loads(user_resp.read().decode())
    scopes = user_resp.headers.get("X-OAuth-Scopes", "none")
    print(f"Authenticated as: {user_data['login']}")
    print(f"Token scopes: {scopes}")
except urllib.error.HTTPError as e:
    body = e.read().decode() if e.fp else ""
    print(f"Auth check failed: {e.code} {e.reason}")
    print(f"Response: {body[:500]}")
    print(f"Headers: {dict(e.headers)}")
    raise

# Check if repo already exists
try:
    check_req = urllib.request.Request(
        "https://api.github.com/repos/skhage/provable-resilience-intelligence",
        headers=gh_headers
    )
    resp = urllib.request.urlopen(check_req, timeout=15)
    repo_data = json.loads(resp.read().decode())
    print(f"\nRepo already exists: {repo_data['html_url']}")
    print(f"Default branch: {repo_data['default_branch']}")
except urllib.error.HTTPError as e:
    if e.code == 404:
        print("\nRepo not found — creating...")
        create_payload = json.dumps({
            "name": "provable-resilience-intelligence",
            "description": "Provable Resilience Intelligence — Governed resilience scoring for Lakelink Fiber enterprise services. DAB-managed Databricks project.",
            "private": False,
            "auto_init": True,
            "has_issues": True,
            "has_projects": True,
            "has_wiki": False
        }).encode()
        create_req = urllib.request.Request(
            "https://api.github.com/user/repos",
            data=create_payload,
            headers={**gh_headers, "Content-Type": "application/json"},
            method="POST"
        )
        create_resp = urllib.request.urlopen(create_req, timeout=30)
        repo = json.loads(create_resp.read().decode())
        print(f"CREATED: {repo['html_url']}")
        print(f"Default branch: {repo['default_branch']}")
        print(f"Clone URL: {repo['clone_url']}")
    else:
        body = e.read().decode() if e.fp else ""
        print(f"Error: {e.code} {e.reason}: {body[:500]}")
        raise

In [0]:
# Create 'develop' branch from main
import urllib.request, urllib.error, json

# Get SHA of main branch HEAD
ref_req = urllib.request.Request(
    "https://api.github.com/repos/skhage/provable-resilience-intelligence/git/ref/heads/main",
    headers=gh_headers
)
ref_resp = urllib.request.urlopen(ref_req, timeout=15)
ref_data = json.loads(ref_resp.read().decode())
main_sha = ref_data["object"]["sha"]
print(f"main HEAD SHA: {main_sha}")

# Create develop branch
try:
    dev_payload = json.dumps({
        "ref": "refs/heads/develop",
        "sha": main_sha
    }).encode()
    dev_req = urllib.request.Request(
        "https://api.github.com/repos/skhage/provable-resilience-intelligence/git/refs",
        data=dev_payload,
        headers=gh_headers,
        method="POST"
    )
    dev_resp = urllib.request.urlopen(dev_req, timeout=15)
    print(f"Created 'develop' branch at {main_sha[:8]}")
except urllib.error.HTTPError as e:
    if e.code == 422:
        print("'develop' branch already exists")
    else:
        raise

In [0]:
# Create a Databricks Repo (Git folder) linked to the GitHub repo
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

try:
    repo = w.repos.create(
        url="https://github.com/skhage/provable-resilience-intelligence.git",
        provider="gitHub",
        path="/Repos/stephen.hage@databricks.com/provable-resilience-intelligence"
    )
    print(f"Databricks Repo created: {repo.path}")
    print(f"Branch: {repo.branch}")
    print(f"URL: {repo.url}")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Databricks Repo already linked.")
    else:
        raise

In [0]:
# Push 9 existing local commits to GitHub.
# Run AFTER the repo exists on GitHub (via Option A or B in Cell 1).

import subprocess

repo_dir = "/Workspace/Users/stephen.hage@databricks.com/provable-resilience-intelligence"

# Check current state
log = subprocess.run(["git", "log", "--oneline"], cwd=repo_dir, capture_output=True, text=True)
print(f"Local commits to push:\n{log.stdout}")

# Push main branch to GitHub
result = subprocess.run(
    ["git", "push", "-u", "origin", "main", "--force"],
    cwd=repo_dir, capture_output=True, text=True
)
print(f"\nPush main: rc={result.returncode}")
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    # Also push develop branch
    result2 = subprocess.run(
        ["git", "push", "-u", "origin", "develop"],
        cwd=repo_dir, capture_output=True, text=True
    )
    print(f"\nPush develop: rc={result2.returncode}")
    if result2.stderr:
        print(result2.stderr)
    print("\n✅ All branches pushed to GitHub!")
    print("   https://github.com/skhage/provable-resilience-intelligence")
else:
    print("\n❌ Push failed. Ensure the repo exists on GitHub first (see Cell 1 instructions).")